In [1]:
#****** Import Libraries *******
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from pathlib import Path
import optuna
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from utils import set_global_seed

/Users/opeolobe/research/dev_eql_fdd/.eqlvenv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set the seed for all random number generators
set_global_seed(42)

In [3]:
# GPU acceleration 
device = torch.device("mps" if torch.backends.mps.is_available() 
                      else "cuda" if torch.cuda.is_available() 
                      else "cpu")

print(f"Using device: {device}")

Using device: mps


In [4]:
#****** Data Loading & Preprocessing ******
project_path = Path.cwd()                         # Get the current working directory
data_path = project_path / "data/normal.xlsx"

df = pd.read_excel(data_path) 

# Drop the first row of the DataFrame (it contains units, not data)
df = df.iloc[1:, :]

# Convert all columns to numeric numpy array of floats
data = df.values.astype(float)  

print("Data shape:", data.shape)

Data shape: (10801, 105)


In [5]:
#************ Train/val split *****************
T = data.shape[0]

# first 80% for training, last 20% for validation
train_size = int(0.97 * T)
X_train = data[:train_size, :]
X_val = data[train_size:, :]

print("Train shape:", X_train.shape)
print("Val shape:", X_val.shape)

Train shape: (10476, 105)
Val shape: (325, 105)


In [6]:
#************** Feature Scaling (Standardization) ***************
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

In [7]:
#************* Tunable Autoencoder Model *************************
class TunableAE(nn.Module):
    def __init__(self, input_dim=105, latent_dim=7, hidden1=64, hidden2=32, activation="Mish"):
        super().__init__()

        act_map = {
            "ReLU": nn.ReLU,
            "LeakyReLU": nn.LeakyReLU,
            "ELU": nn.ELU,
            "GELU": nn.GELU,
            "Mish": nn.Mish,
            "SiLU": nn.SiLU,
            "Tanh": nn.Tanh,
        }
        if activation not in act_map:
            raise ValueError(f"Unsupported activation: {activation}")

        act_fn = act_map[activation]

        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            act_fn(),
            nn.Linear(hidden1, hidden2),
            act_fn(),
            nn.Linear(hidden2, latent_dim),
        )

        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden2),
            act_fn(),
            nn.Linear(hidden2, hidden1),
            act_fn(),
            nn.Linear(hidden1, input_dim),
        )

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat

In [8]:
#******** Hyperparameter Tuning ************
def objective(trial):
    # Hyperparameters search space
    hidden1 = trial.suggest_categorical("hidden1", [64, 96, 128, 160])
    hidden2 = trial.suggest_categorical("hidden2", [16, 32, 48, 64])
    latent_dim = trial.suggest_categorical("latent_dim", [4, 5, 6, 7, 8, 10, 12])

    # reject invalid architecture
    if not (hidden1 > hidden2 > latent_dim):
        raise optuna.exceptions.TrialPruned()

    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128, 256])
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True)
    activation = trial.suggest_categorical("activation", ["ReLU", "LeakyReLU", "GELU", "Mish", "SiLU"])

    # DataLoader setup
    train_dataset = TensorDataset(torch.tensor(X_train_scaled, dtype=torch.float32))
    val_dataset = TensorDataset(torch.tensor(X_val_scaled, dtype=torch.float32))

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    # Model, optimizer, and training loop
    input_dim = X_train_scaled.shape[1]
    model = TunableAE(input_dim=input_dim, latent_dim=latent_dim, hidden1=hidden1, hidden2=hidden2, activation=activation)
    model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    n_epochs = 50

    for epoch in range(n_epochs):
        model.train()
        train_loss = 0.0
        for (x_batch,) in train_loader:
            x_batch = x_batch.to(device)
            # Reconstruction loss
            optimizer.zero_grad()
            x_recon = model(x_batch)
            loss = F.mse_loss(x_recon, x_batch)
            # Backpropagation and optimization step
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * x_batch.size(0)

        # validation for pruning
        model.eval()
        val_sample_errs = []

        with torch.no_grad():
            for (x_batch,) in val_loader:
                x_batch = x_batch.to(device)
                x_recon = model(x_batch)

                # per-sample reconstruction error
                sample_err = torch.mean((x_batch - x_recon) ** 2, dim=1)
                val_sample_errs.append(sample_err.cpu().numpy())


        val_sample_errs = np.concatenate(val_sample_errs)
        val_loss = float(np.mean(val_sample_errs))
        val_err_mean = float(np.mean(val_sample_errs))
        val_err_std = float(np.std(val_sample_errs, ddof=1))
        val_q95 = float(np.quantile(val_sample_errs, 0.95))
        val_q99 = float(np.quantile(val_sample_errs, 0.99))

        # combined pruning metric (main weight on mean val loss, smaller penalty on upper tail)
        prune_metric = val_loss + 0.1 * val_q99

        # Report pruning metric
        trial.report(prune_metric, epoch)

        # Store useful diagnostics
        trial.set_user_attr("last_train_loss", float(train_loss))
        trial.set_user_attr("last_val_loss", float(val_loss))
        trial.set_user_attr("last_val_err_mean", val_err_mean)
        trial.set_user_attr("last_val_err_std", val_err_std)
        trial.set_user_attr("last_val_q95", val_q95)
        trial.set_user_attr("last_val_q99", val_q99)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    # final objective
    return prune_metric

In [9]:
#******** Run Optuna Optimization **********
study = optuna.create_study(
    direction="minimize",
    study_name="AE_Hyperparameter_Tuning",
    pruner=optuna.pruners.MedianPruner(
        n_startup_trials=10,
        n_warmup_steps=10,
        interval_steps=5
    )
)

study.optimize(objective, n_trials=50)

[I 2026-04-06 18:34:23,786] A new study created in memory with name: AE_Hyperparameter_Tuning
[I 2026-04-06 18:34:57,625] Trial 0 finished with value: 0.005820836778730154 and parameters: {'hidden1': 160, 'hidden2': 32, 'latent_dim': 12, 'lr': 0.0032255868692190105, 'batch_size': 32, 'weight_decay': 1.1209896702979982e-06, 'activation': 'ReLU'}. Best is trial 0 with value: 0.005820836778730154.
[I 2026-04-06 18:35:02,584] Trial 1 finished with value: 0.013096493668854237 and parameters: {'hidden1': 128, 'hidden2': 48, 'latent_dim': 5, 'lr': 0.0013940368092675336, 'batch_size': 256, 'weight_decay': 0.0001150873008268117, 'activation': 'ReLU'}. Best is trial 0 with value: 0.005820836778730154.
[I 2026-04-06 18:35:12,823] Trial 2 finished with value: 0.03488216958940029 and parameters: {'hidden1': 128, 'hidden2': 32, 'latent_dim': 5, 'lr': 0.0003484266444174605, 'batch_size': 128, 'weight_decay': 6.874592719922331e-06, 'activation': 'LeakyReLU'}. Best is trial 0 with value: 0.005820836778

In [10]:
print("Best Trial Summary")
best_trial = study.best_trial
print("Best Hyperparameters:")
for k, v in best_trial.params.items():
    print(f"  {k}: {v}")
print("Best objective:", best_trial.value)
print("Best val loss:", best_trial.user_attrs["last_val_loss"])
print("Best val q99:", best_trial.user_attrs["last_val_q99"])
print("Best val std:", best_trial.user_attrs["last_val_err_std"])

Best Trial Summary
Best Hyperparameters:
  hidden1: 160
  hidden2: 32
  latent_dim: 12
  lr: 0.0024869083266652416
  batch_size: 32
  weight_decay: 1.019762838023129e-06
  activation: Mish
Best objective: 0.0022175136720761655
Best val loss: 0.0018419198459014297
Best val q99: 0.0037559382617473602
Best val std: 0.000737003399990499


In [11]:
#***** Save the result *****
study.trials_dataframe().to_csv("optuna_ae_results.csv", index=False)